In [0]:
# notebook 3 - camada GOLD
# aqui a gente monta o star schema (dimensoes, bridges e fato), a tabela de contexto pra Ia
# e responde as 6 perguntas de negocio com display()
# mesma ideia dos notebooks 1 e 2: tudo que e path/schema vem do medalhao_config
import importlib, sys
if 'medalhao_config' in sys.modules:
    del sys.modules['medalhao_config']
from medalhao_config import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(silver_schema, gold_schema)

medalhao_cinema.silver medalhao_cinema.gold


In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime
from pyspark.sql.window import Window


dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

In [0]:
# consertado: tirei daqui a gravacao do df_silver_filmes (isso e da silver, foi pro notebook 2)
# leio as 6 tabelas da silver que a gold precisa (a cotacao ja foi aplicada na silver, entao nao preciso dela aqui)
# nomes das colunas que uso da silver (se na sua silver estiver diferente, ajusta so aqui):
#   tb_generos          -> id_filme, nome_genero
#   tb_pessoas_empresas -> id_filme, nome_entidade, tipo_entidade (Ator/Diretor/Roteirista/Produtora)
def ler_silver(nome):
    return spark.table(table(silver_schema, nome)).withColumn("id_filme", F.col("id_filme").cast("string"))

df_info       = ler_silver("tb_info_filmes")
df_fin        = ler_silver("tb_financeiro_filmes")
df_metricas   = ler_silver("tb_metricas_engajamento")
df_avaliacoes = ler_silver("tb_avaliacoes_usuarios")
df_generos    = ler_silver("tb_generos")
df_pessoas    = ler_silver("tb_pessoas_empresas")

def salvar_gold(df, nome):
    # mesmo padrao de escrita da silver: delta + overwrite
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table(gold_schema, nome))
    print(f"gravado: {table(gold_schema, nome)} ({df.count()} linhas)")

In [0]:
# surrogate key com row_number (Window sem partitionBy = tudo numa particao so, ok pro tamanho dessa base)
w_sk_filme = Window.orderBy("id_filme")

dim_movies = (df_info
    .dropDuplicates(["id_filme"])                                    # garantia extra de 1 linha por filme
    .withColumn("sk_movie_id", F.row_number().over(w_sk_filme).cast("bigint"))
    .select(
        "sk_movie_id",
        F.col("id_filme").cast("string"),                            # chave natural da origem
        F.col("titulo").cast("string"),
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        F.col("duracao_minutos").cast("int"),
        F.col("idioma_original").cast("string"),
        F.col("status_filme").cast("string"),
        F.col("sinopse").cast("string"),
    )
)

dq_check("gold.dim_movies", "sk_movie_id não nulo", dim_movies, F.col("sk_movie_id").isNotNull())
dq_check_unique("gold.dim_movies", "sk_movie_id única", dim_movies, ["sk_movie_id"])
dq_check_unique("gold.dim_movies", "id_filme único", dim_movies, ["id_filme"])

salvar_gold(dim_movies, "dim_movies")
display(dim_movies.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_movies | sk_movie_id não nulo | 0/97879 linhas falharam
[PASS] gold.dim_movies | sk_movie_id única | 0 chaves duplicadas de 97879 linhas
[PASS] gold.dim_movies | id_filme único | 0 chaves duplicadas de 97879 linhas
gravado: medalhao_cinema.gold.dim_movies (97879 linhas)


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,1000004,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
2,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
3,1000007,KYLE BROWNRIGG: INTRODUCING LYLE,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
4,1000011,Worth Your Weight in Gold,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
5,1000014,On va manquer !,2018-05-15,2018,0,fr,Lançado,null
6,1000030,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null
7,1000054,One Hundred Years and Hope,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope."
8,1000058,Homecoming,2023-07-12,2023,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances."
9,1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!"
10,1000073,A Chance To Win,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever."


In [0]:
# antes do join confiro se as chaves tem o mesmo tipo
# depois vejo se alguma referencia ficou sem correspondencia
def validar_fk(filha, pai, chave, rotulo):
    tipo_filha = filha.schema[chave].dataType
    tipo_pai = pai.schema[chave].dataType

    if tipo_filha != tipo_pai:
        raise TypeError(
            f"{rotulo}: tipos diferentes em {chave}: "
            f"{tipo_filha.simpleString()} / {tipo_pai.simpleString()}"
        )

    if filha.filter(F.col(chave).isNull()).limit(1).count():
        raise ValueError(f"{rotulo}: chave nula em {chave}")

    sem_correspondencia = (
        filha.select(chave)
        .join(pai.select(chave), chave, "left_anti")
    )

    if sem_correspondencia.limit(1).count():
        raise ValueError(
            f"{rotulo}: existem valores de {chave} sem correspondencia"
        )

In [0]:
# primeiro deixo um registro por genero
# depois calculo o hash pelo nome, mantendo a mesma padronização
dim_genres = (
    df_generos
    .select(F.trim("nome_genero").alias("nome_genero"))
    .filter(
        F.col("nome_genero").isNotNull()
        & (F.col("nome_genero") != "")
    )
    .distinct()
    .withColumn(
        "sk_genre_id",
        F.sha2(F.col("nome_genero"), 256)
    )
    .select("sk_genre_id", "nome_genero")
)

# confiro se as chaves foram geradas no formato esperado
hash_invalido = (
    F.col("sk_genre_id").isNull()
    | ~F.col("sk_genre_id").rlike("^[0-9a-f]{64}$")
)

if dim_genres.filter(hash_invalido).limit(1).count():
    raise ValueError("existe sk_genre_id nulo ou fora do formato SHA-256")

# cada chave precisa identificar só um registro nessa dimensão
if (
    dim_genres
    .groupBy("sk_genre_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise ValueError("existe sk_genre_id repetido na dimensão")

# faltava essa gravação:
# sem ela o analytics poderia ler a dimensão antiga com ids numericos
salvar_gold(dim_genres, "dim_genres")

# uso a dimensão que realmente ficou gravada
dim_genres = spark.table(table(gold_schema, "dim_genres"))

display(dim_genres)
dim_genres.printSchema()

gravado: medalhao_cinema.gold.dim_genres (329 linhas)


sk_genre_id,nome_genero
271a22f75d73e13fa558d991442ecde8d75a34a0b9f6216d44b116f8612c82dc,Horror
72c31f30a9f6101e9945b6a6c3ccae9175dd7cbe06b2d3ca290a5d1eeae0543e,Trafficking And Intrigue
999f23fcd7bec7075e54bb5dea0d9c548bfe7261f95b911ed8e23d2f4188724f,Identity
068b24a44454e55a8ef11a90858a2bb8084ae4bce35f1aa428fb00e381bb58c2,The Fear Instalation
7333e0d0fa272c44764e53e8685313d127ac188290305fe0a99291bfb07ceff2,To Jay
a412644a300702c8dfc650e4e8b2f8f3abb98d668c5c22294d74e8e8bd7c4f3c,The Partying
deee86ae1da5ee6dbd7693f8caf6abd4516f83204d2c89c2f9bdfc55d1c5659e,She Studies Kendo
508d736ec367c20b9e3ac5d12a501d93bd43bfb7fdfc765d137bfd79378ea5bb,Dance Floor
bae767a93a460ac028ddd9654d9b8020c315e50ed5721b8006c5207272c15d71,Ticking
ff1ccd30a63a82a05861dd4c86e721c96bffa9cf16c105df3caed00c12d68d43,To A Degree


root
 |-- sk_genre_id: string (nullable = true)
 |-- nome_genero: string (nullable = true)



In [0]:
dim_people = (df_pessoas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.trim(F.col("nome_entidade")).alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa"),
    )
    .filter(F.col("nome_pessoa").isNotNull() & (F.col("nome_pessoa") != ""))
    .distinct()
    .withColumn("sk_person_id", F.row_number().over(Window.orderBy("tipo_pessoa", "nome_pessoa")).cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

dq_check("gold.dim_people", "tipo_pessoa só Ator/Diretor/Roteirista", dim_people,
         F.col("tipo_pessoa").isin("Ator", "Diretor", "Roteirista"))
dq_check_unique("gold.dim_people", "nome+tipo único", dim_people, ["nome_pessoa", "tipo_pessoa"])

salvar_gold(dim_people, "dim_people")
display(dim_people.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_people | tipo_pessoa só Ator/Diretor/Roteirista | 0/419419 linhas falharam
[PASS] gold.dim_people | nome+tipo único | 0 chaves duplicadas de 419419 linhas
gravado: medalhao_cinema.gold.dim_people (419419 linhas)


sk_person_id,nome_pessoa,tipo_pessoa
1,"""1969 (\""""careful With That Axe""""""",Ator
2,'ana Ika,Ator
3,'e-gotti' Eric Johnson,Ator
4,'jeeva' Ravi,Ator
5,'meesai' Mohan,Ator
6,'meesai' Rajendran,Ator
7,'om' Rakesh Chaturvedi,Ator
8,'poo' Ram,Ator
9,'sunday Jeff' Silverman,Ator
10,/1pvyiul8yq9imhmjk2fkx9qp4zf.jpg,Ator


In [0]:
dim_companies = (df_pessoas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.trim(F.col("nome_entidade")).alias("nome_produtora"))
    .filter(F.col("nome_produtora").isNotNull() & (F.col("nome_produtora") != ""))
    .distinct()
    .withColumn("sk_company_id", F.row_number().over(Window.orderBy("nome_produtora")).cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)

dq_check_unique("gold.dim_companies", "nome_produtora único", dim_companies, ["nome_produtora"])

salvar_gold(dim_companies, "dim_companies")
display(dim_companies.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_companies | nome_produtora único | 0 chaves duplicadas de 45177 linhas
gravado: medalhao_cinema.gold.dim_companies (45177 linhas)


sk_company_id,nome_produtora
1,"""14 April 1969 (\""""afternoon\""""\"""""""""""
2,"""\"""" And Many More.\"""""""""""
3,"""\"""" \""""\""""this Is It"""""""
4,"""\"""" \""""\""""where I Come From"""""""
5,"""\""""\""""green""""\"""" Productions""""""""canadaenglish"
6,"""\""""\""""we W͟e͟r͟e͟ Amazing""""""we A͟r͟e͟ Amazing\\""""\"""""""""""
7,"""\""""\""""yeni Film""""\"""" Film Company""""""""""hebrew"
8,"""\""""the First Cigarette\""""\"""""""""""
9,"""\""""when I Get Home\""""\"""""""""""
10,"""\creative Association """"\""""avos""""\""""\""""""""russiarussian"


In [0]:
dim_reviews = (df_avaliacoes
    .groupBy("id_filme")
    .agg(
        F.count(F.lit(1)).cast("int").alias("qtd_avaliacoes_usuarios"),     # conta todas as avaliacoes
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"),  # avg ja ignora NULL
    )
    .join(dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner")  # so filmes que existem na dim
    .withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")).cast("bigint"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)

dq_check_unique("gold.dim_reviews", "1 linha por filme", dim_reviews, ["sk_movie_id"])
dq_check("gold.dim_reviews", "nota média entre 0 e 10", dim_reviews,
         F.col("nota_media_usuarios").isNull() | F.col("nota_media_usuarios").between(0, 10))

salvar_gold(dim_reviews, "dim_reviews")
display(dim_reviews.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_reviews | 1 linha por filme | 0 chaves duplicadas de 27303 linhas
[PASS] gold.dim_reviews | nota média entre 0 e 10 | 0/27303 linhas falharam
gravado: medalhao_cinema.gold.dim_reviews (27303 linhas)


sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
1,1,1,0.1
2,5,1,3.3
3,8,1,6.8
4,17,1,5.4
5,19,1,5.3
6,20,1,4.4
7,23,3,4.75
8,24,1,4.8
9,26,1,6.6
10,33,1,7.5


escolhi usar o sha2 pq ele gera a mesma chave quando recebe o mesmo valor de entrada. assim, quando a base crescer, os registros existentes continuam com a mesma chave, desde que o campo usado no hash e sua padronização nao mudem.

o monotonically_increasing_id nao é deterministico, pq depende das partições do spark. o problema nao é inserir novos dados, mas recalcular os ids e acabar mudando as ligações entre dimensões, fato e bridges.

comparado ao row_number com ordenação global, o sha2 permite calcular as chaves por registro sem ordenar a base inteira só pra numerar as linhas. isso pode melhorar o desempenho nessa etapa, mas precisa ser medido no pipeline.

nao significa que o processamento inteiro seja O(1). o custo do hash depende do tamanho da entrada e preciso calcular ele pra todos os registros. também nao garante que os joins pra buscar as FKs fiquem mais rapidos, pq eles continuam tendo seus próprios custos.

tem ainda uma diferença em relação ao enunciado: o sha2 retorna STRING, enquanto as chaves foram especificadas como BIGINT. essa mudança de tipo precisa ser alinhada pra entrega.

referencia: [documentação do sha2](https://spark.apache.org/docs/3.5.6/api/python/reference/pyspark.sql/api/pyspark.sql.functions.sha2.html)

## Tabelas ponte (bridge)

Um filme tem N generos / N pessoas / N produtoras. Se eu ligasse isso direto na fato, cada filme apareceria varias vezes e a soma de receita dobraria.
Por isso cada bridge guarda so o par de chaves (`sk_movie_id`, `sk_xxx_id`), 1 linha por par (`distinct()`).
Passo a passo: pego a tabela silver (id_filme + nome) -> join com `dim_movies` pra achar `sk_movie_id` -> join com a dimensao pra achar a outra sk.

In [0]:
# filme x genero
# busco o hash na dimensão, nao calculo outra chave na bridge
bridge_movie_genre = (
    df_generos
    .select(
        "id_filme",
        F.trim("nome_genero").alias("nome_genero")
    )
    .join(
        dim_movies.select("sk_movie_id", "id_filme"),
        "id_filme",
        "inner"
    )
    .join(dim_genres, "nome_genero", "inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

# filme x pessoa
# nome e tipo juntos identificam a pessoa nesse modelo
bridge_movie_person = (
    df_pessoas
    .filter(
        F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")
    )
    .select(
        "id_filme",
        F.trim("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .join(
        dim_movies.select("sk_movie_id", "id_filme"),
        "id_filme",
        "inner"
    )
    .join(
        dim_people,
        ["nome_pessoa", "tipo_pessoa"],
        "inner"
    )
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

# filme x produtora
bridge_movie_company = (
    df_pessoas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        "id_filme",
        F.trim("nome_entidade").alias("nome_produtora")
    )
    .join(
        dim_movies.select("sk_movie_id", "id_filme"),
        "id_filme",
        "inner"
    )
    .join(dim_companies, "nome_produtora", "inner")
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

# confiro tipos, nulos e referencias das três bridges
for nome, bridge, dimensao, chave in [
    (
        "bridge_movie_genre",
        bridge_movie_genre,
        dim_genres,
        "sk_genre_id"
    ),
    (
        "bridge_movie_person",
        bridge_movie_person,
        dim_people,
        "sk_person_id"
    ),
    (
        "bridge_movie_company",
        bridge_movie_company,
        dim_companies,
        "sk_company_id"
    ),
]:
        # registro as checagens pra aparecer no historico de qualidade
    dq_check_unique(
        f"gold.{nome}",
        "1 linha por par de chaves",
        bridge,
        ["sk_movie_id", chave]
    )

    dq_check(
        f"gold.{nome}",
        "chaves não nulas",
        bridge,
        F.col("sk_movie_id").isNotNull()
        & F.col(chave).isNotNull()
    )

    # os dois resultados acima pertencem à bridge dessa iteração
    if any(not resultado.passed for resultado in dq_results[-2:]):
        raise ValueError(f"[FAIL] {nome}: falha nas checagens de qualidade")

    # confiro também os tipos e se as referencias existem nas dimensões
    validar_fk(bridge, dim_movies, "sk_movie_id", nome)
    validar_fk(bridge, dimensao, chave, nome)

    # o mesmo filme pode ter varios vinculos, mas o par nao pode repetir
    if (
        bridge
        .groupBy("sk_movie_id", chave)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    ):
        raise ValueError(f"{nome}: par de chaves repetido")
salvar_gold(bridge_movie_genre, "bridge_movie_genre")
salvar_gold(bridge_movie_person, "bridge_movie_person")
salvar_gold(bridge_movie_company, "bridge_movie_company")

display(bridge_movie_genre.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.bridge_movie_genre | 1 linha por par de chaves | 0 chaves duplicadas de 131104 linhas


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.bridge_movie_genre | chaves não nulas | 0/131104 linhas falharam
[PASS] gold.bridge_movie_person | 1 linha por par de chaves | 0 chaves duplicadas de 777050 linhas
[PASS] gold.bridge_movie_person | chaves não nulas | 0/777050 linhas falharam
[PASS] gold.bridge_movie_company | 1 linha por par de chaves | 0 chaves duplicadas de 117780 linhas
[PASS] gold.bridge_movie_company | chaves não nulas | 0/117780 linhas falharam
gravado: medalhao_cinema.gold.bridge_movie_genre (131104 linhas)
gravado: medalhao_cinema.gold.bridge_movie_person (777050 linhas)
gravado: medalhao_cinema.gold.bridge_movie_company (117780 linhas)


sk_movie_id,sk_genre_id
11,64cff1319d2fd2cbb7a1e84ccecf22c1cc07b24435cdb522f8c0aa525d6002a6
28,271a22f75d73e13fa558d991442ecde8d75a34a0b9f6216d44b116f8612c82dc
98,542855513e4f74d08a92352680c339d82e2ae347fb021a7128c7f4146c40d295
114,da72bf5bbca4ca33b822955cfbbf9e5b84e4e0fa5e5e68cf9d30054109fac58b
115,64cff1319d2fd2cbb7a1e84ccecf22c1cc07b24435cdb522f8c0aa525d6002a6
117,da72bf5bbca4ca33b822955cfbbf9e5b84e4e0fa5e5e68cf9d30054109fac58b
124,da72bf5bbca4ca33b822955cfbbf9e5b84e4e0fa5e5e68cf9d30054109fac58b
132,da72bf5bbca4ca33b822955cfbbf9e5b84e4e0fa5e5e68cf9d30054109fac58b
142,35624b8800fae3fd49027ffc8d2a4e4bd697a0166c97ec564b5ea19b10b242dd
160,9a3c16424750cbde029c47e059f219dc64d8943ac4073a1cc5bc9accd979fc17


## gold.fact_movies_performance

Grao = 1 filme. Parto da `dim_movies` (que ja e unica) e faco LEFT JOIN com financeiro e metricas.
Antes do join tiro duplicata por `id_filme` nas duas tabelas: se a silver tiver o mesmo filme 2x, o join dobraria a linha e quebraria o grao.
LEFT JOIN porque filme sem financeiro/metricas continua na fato (com NULL), nao pode sumir.

In [0]:
fin = df_fin.dropDuplicates(["id_filme"])
met = df_metricas.dropDuplicates(["id_filme"])

# A fato deve conter somente filmes já lançados.
filmes_lancados = (
    dim_movies
    .filter(
        (F.col("status_filme") == "Lançado")
        & F.col("data_lancamento").isNotNull()
        & (F.col("data_lancamento") <= F.current_date())
    )
    .select("sk_movie_id", "id_filme")
)

# LEFT JOIN preserva filmes sem informações financeiras ou de engajamento.
fact_movies_performance = (
    filmes_lancados
    .join(fin, "id_filme", "left")
    .join(met, "id_filme", "left")
    .select(
        "sk_movie_id",

        # Métricas financeiras.
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),

        # Métricas de engajamento.
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)

dq_check_unique(
    "gold.fact_movies_performance",
    "1 linha por filme (grão)",
    fact_movies_performance,
    ["sk_movie_id"]
)

qtd_lancados = filmes_lancados.count()
qtd_fato = fact_movies_performance.count()

print(
    "Filmes lançados:", qtd_lancados,
    "| Linhas na fato:", qtd_fato
)

# Interrompe antes de gravar se os joins alterarem a quantidade de filmes.
if qtd_fato != qtd_lancados:
    raise ValueError(
        f"Quantidade inesperada na fato: {qtd_fato}. "
        f"Esperado: {qtd_lancados} filmes lançados."
    )

salvar_gold(fact_movies_performance, "fact_movies_performance")
display(fact_movies_performance.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.fact_movies_performance | 1 linha por filme (grão) | 0 chaves duplicadas de 93993 linhas
Filmes lançados: 93993 | Linhas na fato: 93993
gravado: medalhao_cinema.gold.fact_movies_performance (93993 linhas)


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1,null,null,null,null,null,null,1.132,0.0,0,6.8,27
2,null,null,null,null,null,null,0.6,0.0,0,null,40
3,null,null,null,null,null,null,0.6,0.0,0,4.7,10
4,null,null,null,null,null,null,1.169,0.0,0,4.8,16
5,null,null,null,null,null,null,0.6,0.0,0,7.2,15
6,null,null,null,null,null,null,0.615,0.0,0,null,25
7,null,null,null,null,null,null,0.6,0.0,0,5.9,10
8,4700000.00,null,null,23742050.00,null,null,1.489,6.75,6,6.2,382
9,null,null,null,null,null,null,0.6,0.0,0,7.7,25
10,6000000.00,null,null,30309000.00,null,null,13.212,6.8,15,null,236


## gold.dim_movies

In [0]:
# a silver precisa entregar um registro por filme
# valido antes de gerar a chave pra nao esconder duplicatas
dq_check(
    "silver.tb_info_filmes",
    "id_filme preenchido",
    df_info,
    F.col("id_filme").isNotNull()
    & (F.trim("id_filme") != "")
)

dq_check_unique(
    "silver.tb_info_filmes",
    "1 linha por filme",
    df_info,
    ["id_filme"]
)

if any(not resultado.passed for resultado in dq_results[-2:]):
    raise ValueError("corrigir os ids dos filmes na silver antes de continuar")

dim_movies = (
    df_info
    .withColumn(
        "sk_movie_id",
        F.row_number()
        .over(Window.orderBy("id_filme"))
        .cast("bigint")
    )
    .select(
        "sk_movie_id",
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("titulo").cast("string").alias("titulo"),
        F.col("data_lancamento").cast("date").alias("data_lancamento"),
        F.col("ano_lancamento").cast("int").alias("ano_lancamento"),
        F.col("duracao_minutos").cast("int").alias("duracao_minutos"),
        F.col("idioma_original").cast("string").alias("idioma_original"),
        F.col("status_filme").cast("string").alias("status_filme"),
        F.col("sinopse").cast("string").alias("sinopse")
    )
)

dq_check_unique(
    "gold.dim_movies", "PK única",
    dim_movies, ["sk_movie_id"]
)

salvar_gold(dim_movies, "dim_movies")
display(dim_movies.limit(10))

[PASS] silver.tb_info_filmes | id_filme preenchido | 0/97879 linhas falharam
[PASS] silver.tb_info_filmes | 1 linha por filme | 0 chaves duplicadas de 97879 linhas


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_movies | PK única | 0 chaves duplicadas de 97879 linhas
gravado: medalhao_cinema.gold.dim_movies (97879 linhas)


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,1000004,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
2,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
3,1000007,KYLE BROWNRIGG: INTRODUCING LYLE,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
4,1000011,Worth Your Weight in Gold,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
5,1000014,On va manquer !,2018-05-15,2018,0,fr,Lançado,null
6,1000030,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null
7,1000054,One Hundred Years and Hope,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope."
8,1000058,Homecoming,2023-07-12,2023,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances."
9,1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!"
10,1000073,A Chance To Win,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever."


gold.dim_genres


In [0]:
# aqui monto o catalogo de generos
# os nomes precisam chegar validados da silver
# tirar espaços e duplicatas nao corrige um pais cadastrado como genero
dim_genres = (
    df_generos
    .select(F.trim("nome_genero").alias("nome_genero"))
    .filter(
        F.col("nome_genero").isNotNull()
        & (F.col("nome_genero") != "")
    )
    .distinct()
    .withColumn(
        "sk_genre_id",
        F.row_number()
        .over(Window.orderBy("nome_genero"))
        .cast("bigint")
    )
    .select("sk_genre_id", "nome_genero")
)

dq_check_unique(
    "gold.dim_genres", "nome de gênero único",
    dim_genres, ["nome_genero"]
)

dq_check_unique(
    "gold.dim_genres", "PK única",
    dim_genres, ["sk_genre_id"]
)

salvar_gold(dim_genres, "dim_genres")
display(dim_genres)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_genres | nome de gênero único | 0 chaves duplicadas de 329 linhas
[PASS] gold.dim_genres | PK única | 0 chaves duplicadas de 329 linhas
gravado: medalhao_cinema.gold.dim_genres (329 linhas)


sk_genre_id,nome_genero
1,A Bored Housewife
2,A Country That
3,A Father
4,A Mother Hen
5,A Printer
6,A Riveting
7,A Scraggly Rascal
8,A Tantric Masseuse
9,A Tyrant
10,Abby


gold.dim_people

In [0]:
# separo as pessoas das produtoras
# nesse modelo a combinação nome + tipo identifica cada registro
dim_people = (
    df_pessoas
    .filter(
        F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")
    )
    .select(
        F.trim("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").cast("string").alias("tipo_pessoa")
    )
    .filter(
        F.col("nome_pessoa").isNotNull()
        & (F.col("nome_pessoa") != "")
    )
    .distinct()
    .withColumn(
        "sk_person_id",
        F.row_number()
        .over(Window.orderBy("tipo_pessoa", "nome_pessoa"))
        .cast("bigint")
    )
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

dq_check_unique(
    "gold.dim_people", "nome + tipo único",
    dim_people, ["nome_pessoa", "tipo_pessoa"]
)

dq_check(
    "gold.dim_people", "tipo permitido",
    dim_people,
    F.col("tipo_pessoa").isin("Ator", "Diretor", "Roteirista")
)

salvar_gold(dim_people, "dim_people")
display(dim_people.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_people | nome + tipo único | 0 chaves duplicadas de 419419 linhas
[PASS] gold.dim_people | tipo permitido | 0/419419 linhas falharam
gravado: medalhao_cinema.gold.dim_people (419419 linhas)


sk_person_id,nome_pessoa,tipo_pessoa
1,"""1969 (\""""careful With That Axe""""""",Ator
2,'ana Ika,Ator
3,'e-gotti' Eric Johnson,Ator
4,'jeeva' Ravi,Ator
5,'meesai' Mohan,Ator
6,'meesai' Rajendran,Ator
7,'om' Rakesh Chaturvedi,Ator
8,'poo' Ram,Ator
9,'sunday Jeff' Silverman,Ator
10,/1pvyiul8yq9imhmjk2fkx9qp4zf.jpg,Ator


In [0]:
#gold.dim_companies

In [0]:
# monto o catalogo de produtoras sem repetir o nome
dim_companies = (
    df_pessoas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.trim("nome_entidade").alias("nome_produtora"))
    .filter(
        F.col("nome_produtora").isNotNull()
        & (F.col("nome_produtora") != "")
    )
    .distinct()
    .withColumn(
        "sk_company_id",
        F.row_number()
        .over(Window.orderBy("nome_produtora"))
        .cast("bigint")
    )
    .select("sk_company_id", "nome_produtora")
)

dq_check_unique(
    "gold.dim_companies", "nome de produtora único",
    dim_companies, ["nome_produtora"]
)

salvar_gold(dim_companies, "dim_companies")
display(dim_companies.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_companies | nome de produtora único | 0 chaves duplicadas de 45177 linhas
gravado: medalhao_cinema.gold.dim_companies (45177 linhas)


sk_company_id,nome_produtora
1,"""14 April 1969 (\""""afternoon\""""\"""""""""""
2,"""\"""" And Many More.\"""""""""""
3,"""\"""" \""""\""""this Is It"""""""
4,"""\"""" \""""\""""where I Come From"""""""
5,"""\""""\""""green""""\"""" Productions""""""""canadaenglish"
6,"""\""""\""""we W͟e͟r͟e͟ Amazing""""""we A͟r͟e͟ Amazing\\""""\"""""""""""
7,"""\""""\""""yeni Film""""\"""" Film Company""""""""""hebrew"
8,"""\""""the First Cigarette\""""\"""""""""""
9,"""\""""when I Get Home\""""\"""""""""""
10,"""\creative Association """"\""""avos""""\""""\""""""""russiarussian"


gold.dim_reviews

In [0]:
# resumo as avaliações em uma linha por filme
# conto todas as avaliações e calculo a média das notas preenchidas
dim_reviews = (
    df_avaliacoes
    .groupBy("id_filme")
    .agg(
        F.count(F.lit(1)).cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2)
        .cast("double")
        .alias("nota_media_usuarios")
    )
    .join(
        dim_movies.select("id_filme", "sk_movie_id"),
        "id_filme",
        "inner"
    )
    .withColumn(
        "sk_review_id",
        F.row_number()
        .over(Window.orderBy("sk_movie_id"))
        .cast("bigint")
    )
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

dq_check_unique(
    "gold.dim_reviews", "1 linha por filme",
    dim_reviews, ["sk_movie_id"]
)

dq_check(
    "gold.dim_reviews", "nota média entre 0 e 10 ou ausente",
    dim_reviews,
    F.col("nota_media_usuarios").isNull()
    | F.col("nota_media_usuarios").between(0, 10)
)

if any(not resultado.passed for resultado in dq_results[-2:]):
    raise ValueError("falha na qualidade das avaliações resumidas")

validar_fk(
    dim_reviews, dim_movies,
    "sk_movie_id", "avaliações -> filmes"
)

salvar_gold(dim_reviews, "dim_reviews")
display(dim_reviews.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_reviews | 1 linha por filme | 0 chaves duplicadas de 27303 linhas
[PASS] gold.dim_reviews | nota média entre 0 e 10 ou ausente | 0/27303 linhas falharam
gravado: medalhao_cinema.gold.dim_reviews (27303 linhas)


sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
1,1,1,0.1
2,5,1,3.3
3,8,1,6.8
4,17,1,5.4
5,19,1,5.3
6,20,1,4.4
7,23,3,4.75
8,24,1,4.8
9,26,1,6.6
10,33,1,7.5


gold.fact_movies_performance

In [0]:
# financeiro e metricas precisam ter no maximo uma linha por filme
# se tiver duplicata, paro pra corrigir na silver
for nome, df in [
    ("silver.tb_financeiro_filmes", df_fin),
    ("silver.tb_metricas_engajamento", df_metricas)
]:
    dq_check_unique(nome, "1 linha por filme", df, ["id_filme"])

    if not dq_results[-1].passed:
        raise ValueError(f"{nome}: corrigir duplicatas antes do join")

# considero somente lançamentos realizados
filmes_lancados = (
    dim_movies
    .filter(
        (F.col("status_filme") == "Lançado")
        & F.col("data_lancamento").isNotNull()
        & (F.col("data_lancamento") <= F.current_date())
    )
    .select("sk_movie_id", "id_filme")
)

# left join mantém o filme mesmo quando faltam métricas
fact_movies_performance = (
    filmes_lancados
    .join(df_fin, "id_filme", "left")
    .join(df_metricas, "id_filme", "left")
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        F.col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        F.col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        F.col("receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        F.col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),
        F.col("popularidade").cast("double").alias("popularidade"),
        F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        F.col("qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        F.col("qtd_votos_imdb").cast("int").alias("qtd_votos_imdb")
    )
)

dq_check_unique(
    "gold.fact_movies_performance",
    "1 linha por filme",
    fact_movies_performance,
    ["sk_movie_id"]
)

if not dq_results[-1].passed:
    raise ValueError("a fato ficou com filmes repetidos")

validar_fk(
    fact_movies_performance, dim_movies,
    "sk_movie_id", "fato -> filmes"
)

qtd_esperada = filmes_lancados.count()
qtd_fato = fact_movies_performance.count()

if qtd_fato != qtd_esperada:
    raise ValueError(
        f"fato com {qtd_fato} linhas; esperado: {qtd_esperada}"
    )

print(f"[PASS] fato preservou os {qtd_esperada} filmes lançados")

salvar_gold(fact_movies_performance, "fact_movies_performance")
display(fact_movies_performance.limit(10))

[PASS] silver.tb_financeiro_filmes | 1 linha por filme | 0 chaves duplicadas de 99006 linhas
[PASS] silver.tb_metricas_engajamento | 1 linha por filme | 0 chaves duplicadas de 99013 linhas


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.fact_movies_performance | 1 linha por filme | 0 chaves duplicadas de 93993 linhas
[PASS] fato preservou os 93993 filmes lançados
gravado: medalhao_cinema.gold.fact_movies_performance (93993 linhas)


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1,null,null,null,null,null,null,1.132,0.0,0,6.8,27
2,null,null,null,null,null,null,0.6,0.0,0,null,40
3,null,null,null,null,null,null,0.6,0.0,0,4.7,10
4,null,null,null,null,null,null,1.169,0.0,0,4.8,16
5,null,null,null,null,null,null,0.6,0.0,0,7.2,15
6,null,null,null,null,null,null,0.615,0.0,0,null,25
7,null,null,null,null,null,null,0.6,0.0,0,5.9,10
8,4700000.00,null,null,23742050.00,null,null,1.489,6.75,6,6.2,382
9,null,null,null,null,null,null,0.6,0.0,0,7.7,25
10,6000000.00,null,null,30309000.00,null,null,13.212,6.8,15,null,236


gold.bridge_movie_genre

In [0]:
# cada linha representa uma ligação entre filme e genero
bridge_movie_genre = (
    df_generos
    .select(
        "id_filme",
        F.trim("nome_genero").alias("nome_genero")
    )
    .join(
        dim_movies.select("id_filme", "sk_movie_id"),
        "id_filme",
        "inner"
    )
    .join(dim_genres, "nome_genero", "inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

dq_check_unique(
    "gold.bridge_movie_genre",
    "1 linha por par filme/gênero",
    bridge_movie_genre,
    ["sk_movie_id", "sk_genre_id"]
)

if not dq_results[-1].passed:
    raise ValueError("par filme/gênero repetido")

validar_fk(
    bridge_movie_genre, dim_movies,
    "sk_movie_id", "bridge gênero -> filmes"
)

validar_fk(
    bridge_movie_genre, dim_genres,
    "sk_genre_id", "bridge gênero -> gêneros"
)

salvar_gold(bridge_movie_genre, "bridge_movie_genre")
display(bridge_movie_genre.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.bridge_movie_genre | 1 linha por par filme/gênero | 0 chaves duplicadas de 131104 linhas
gravado: medalhao_cinema.gold.bridge_movie_genre (131104 linhas)


sk_movie_id,sk_genre_id
57126,1
81331,2
91443,3
23232,4
69195,5
28702,6
75002,8
90105,9
62187,10
11802,11


gold.bridge_movie_person

In [0]:
# busco a pessoa pelo nome e pelo papel que ela exerce
bridge_movie_person = (
    df_pessoas
    .filter(
        F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")
    )
    .select(
        "id_filme",
        F.trim("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .join(
        dim_movies.select("id_filme", "sk_movie_id"),
        "id_filme",
        "inner"
    )
    .join(dim_people, ["nome_pessoa", "tipo_pessoa"], "inner")
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

dq_check_unique(
    "gold.bridge_movie_person",
    "1 linha por par filme/pessoa",
    bridge_movie_person,
    ["sk_movie_id", "sk_person_id"]
)

if not dq_results[-1].passed:
    raise ValueError("par filme/pessoa repetido")

validar_fk(
    bridge_movie_person, dim_movies,
    "sk_movie_id", "bridge pessoa -> filmes"
)

validar_fk(
    bridge_movie_person, dim_people,
    "sk_person_id", "bridge pessoa -> pessoas"
)

salvar_gold(bridge_movie_person, "bridge_movie_person")
display(bridge_movie_person.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.bridge_movie_person | 1 linha por par filme/pessoa | 0 chaves duplicadas de 777050 linhas
gravado: medalhao_cinema.gold.bridge_movie_person (777050 linhas)


sk_movie_id,sk_person_id
40317,131607
45519,138100
65848,9839
78274,88755
87220,44084
15287,247955
23092,28094
82594,45736
95790,139466
17993,28009


gold.bridge_movie_company

In [0]:
# um filme pode ter varias produtoras
# guardo cada ligação uma vez, sem levar esses vinculos pra fato
bridge_movie_company = (
    df_pessoas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        "id_filme",
        F.trim("nome_entidade").alias("nome_produtora")
    )
    .join(
        dim_movies.select("id_filme", "sk_movie_id"),
        "id_filme",
        "inner"
    )
    .join(dim_companies, "nome_produtora", "inner")
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

dq_check_unique(
    "gold.bridge_movie_company",
    "1 linha por par filme/produtora",
    bridge_movie_company,
    ["sk_movie_id", "sk_company_id"]
)

if not dq_results[-1].passed:
    raise ValueError("par filme/produtora repetido")

validar_fk(
    bridge_movie_company, dim_movies,
    "sk_movie_id", "bridge produtora -> filmes"
)

validar_fk(
    bridge_movie_company, dim_companies,
    "sk_company_id", "bridge produtora -> produtoras"
)

salvar_gold(bridge_movie_company, "bridge_movie_company")
display(bridge_movie_company.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.bridge_movie_company | 1 linha por par filme/produtora | 0 chaves duplicadas de 117780 linhas
gravado: medalhao_cinema.gold.bridge_movie_company (117780 linhas)


sk_movie_id,sk_company_id
31546,1
29410,2
67102,3
48585,4
53512,5
96153,6
53548,7
66395,8
52078,9
32265,10


## Desafio de Analytics

Daqui pra baixo so leitura das tabelas gold gravadas. Cada pergunta = 1 `display()`.

In [0]:
# releio as tabelas gravadas na gold pra responder o desafio
fato = spark.table(table(gold_schema, "fact_movies_performance"))
filmes = spark.table(table(gold_schema, "dim_movies"))
generos = spark.table(table(gold_schema, "dim_genres"))
pessoas = spark.table(table(gold_schema, "dim_people"))
empresas = spark.table(table(gold_schema, "dim_companies"))

br_genero = spark.table(table(gold_schema, "bridge_movie_genre"))
br_pessoa = spark.table(table(gold_schema, "bridge_movie_person"))
br_empresa = spark.table(table(gold_schema, "bridge_movie_company"))

# o enunciado pede as chaves como BIGINT
# verifico antes dos joins pra identificar alguma tabela antiga
for nome, df, chaves in [
    ("fact_movies_performance", fato, ["sk_movie_id"]),
    ("dim_movies", filmes, ["sk_movie_id"]),
    ("dim_genres", generos, ["sk_genre_id"]),
    ("dim_people", pessoas, ["sk_person_id"]),
    ("dim_companies", empresas, ["sk_company_id"]),
    ("bridge_movie_genre", br_genero, ["sk_movie_id", "sk_genre_id"]),
    ("bridge_movie_person", br_pessoa, ["sk_movie_id", "sk_person_id"]),
    ("bridge_movie_company", br_empresa, ["sk_movie_id", "sk_company_id"]),
]:
    for chave in chaves:
        tipo = df.schema[chave].dataType.simpleString()

        if tipo != "bigint":
            raise TypeError(
                f"{nome}.{chave}: esperado bigint, encontrado {tipo}. "
                "Reexecute a criação e gravação das tabelas relacionadas."
            )

print("[PASS] tipos das chaves usadas no analytics")

# confiro se as referencias persistidas existem
validar_fk(fato, filmes, "sk_movie_id", "fato -> filmes")

for nome, bridge, dimensao, chave in [
    ("gêneros", br_genero, generos, "sk_genre_id"),
    ("pessoas", br_pessoa, pessoas, "sk_person_id"),
    ("produtoras", br_empresa, empresas, "sk_company_id"),
]:
    validar_fk(bridge, filmes, "sk_movie_id", f"bridge {nome} -> filmes")
    validar_fk(bridge, dimensao, chave, f"bridge {nome} -> dimensão")

print("[PASS] referencias das tabelas usadas no analytics")

[PASS] tipos das chaves usadas no analytics
[PASS] referencias das tabelas usadas no analytics


### 1. Receita total (R$) de todos os filmes da base

In [0]:
# somo direto na fato, que foi validada com uma linha por filme lançado
# receitas nulas nao entram na soma
display(
    fato.agg(
        F.sum("receita_brl").alias("receita_total_brl")
    )
)

receita_total_brl
790326807877.48


### 2. Top 5 filmes por popularidade

In [0]:
# junto com a dimensão pra buscar o titulo do filme
# tiro popularidade nula e ordeno do maior valor pro menor
# se empatar, uso o id pra manter a escolha dos 5 consistente
display(
    fato.join(filmes, "sk_movie_id")
    .filter(F.col("popularidade").isNotNull())
    .select("sk_movie_id", "titulo", "popularidade")
    .orderBy(
        F.col("popularidade").desc(),
        F.col("sk_movie_id").asc()
    )
    .limit(5)
    .select("titulo", "popularidade") # essa foi chata pra resolver 
)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
retribution,1547.22


## ignorar isso foi para fazer um teste antes de voltar pra prata... 

In [0]:
alvo = ( # la vamos nos 
    spark.table(table(silver_schema, "tb_info_filmes"))
    .filter(F.lower(F.trim(F.col("titulo"))) == "battipaglia 1969")
    .select("id_filme", "titulo", "data_lancamento")
)

display(
    spark.table(table(bronze_schema, "tb_movies_metrics"))
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
    .join(alvo, "id_filme", "inner")
    .select(
        "id_filme",
        "titulo",
        "data_lancamento",
        "popularity",
        "vote_average",
        "vote_count",
        "averageRating",
        "numVotes",
        "ingestion_datetime"
    )
)

id_filme,titulo,data_lancamento,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
418746,Battipaglia 1969,2016-09-30,1969,10.0,null,Italian,null,2026-10-16T04:01:56.948Z
418746,Battipaglia 1969,2016-09-30,1969,10.0,null,Italian,null,2026-10-16T03:11:37.231Z
418746,Battipaglia 1969,2016-09-30,1969,10.0,null,Italian,null,2026-10-16T03:03:10.405Z
418746,Battipaglia 1969,2016-09-30,1969,10.0,null,Italian,null,2026-10-16T03:01:46.508Z
418746,Battipaglia 1969,2016-09-30,1969,10.0,null,Italian,null,2026-10-16T02:01:59.651Z
418746,Battipaglia 1969,2016-09-30,1969,10.0,null,Italian,null,2026-10-16T01:01:54.045Z
418746,Battipaglia 1969,2016-09-30,1969,10.0,null,Italian,null,2026-10-16T00:02:00.571Z
418746,Battipaglia 1969,2016-09-30,1969,10.0,null,Italian,null,2026-10-15T23:01:57.643Z
418746,Battipaglia 1969,2016-09-30,1969,10.0,null,Italian,null,2026-10-15T22:01:49.115Z
418746,Battipaglia 1969,2016-09-30,1969,10.0,null,Italian,null,2026-10-15T21:01:50.865Z


### 3. Quantidade de filmes por gênero (maior -> menor)

In [0]:
# a bridge tem uma linha por par filme e genero
# junto com a dimensão pra pegar o nome
# conto filmes distintos dentro de cada genero e ordeno pela quantidade
display(
    br_genero
    .join(generos, "sk_genre_id")
    .groupBy("nome_genero")
    .agg(
        F.countDistinct("sk_movie_id").alias("qtd_filmes")
    )
    .orderBy(
        F.col("qtd_filmes").desc(),
        F.col("nome_genero")
    )
)

nome_genero,qtd_filmes
Drama,30383
Documentary,18599
Comedy,17296
Thriller,9397
Horror,9134
Romance,6968
Action,5513
Crime,4304
Animation,4142
Tv Movie,3665


### 4. Top 10 filmes por receita com RANK()

In [0]:
w_rank = Window.orderBy(F.col("receita_usd").desc())
# aqui uso rank pela receita entao filmes com a mesma receita recebem a mesma posição
# depois ordeno os empatados pelo id do filme só pra escolher sempre do mesmo jeito
# e limito em 10 pq o enunciado pediu 10 filmes
# nao coloco o id dentro da janela do rank pq isso desfaz o empate
display(
    fato.join(filmes, "sk_movie_id")
    .filter(F.col("receita_usd").isNotNull())
    .withColumn("posicao", F.rank().over(w_rank))
    .orderBy("posicao", "sk_movie_id")
    .limit(10)
    .select("posicao", "titulo", "receita_usd", "receita_brl")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posicao,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,14144200000.00
2,Avatar: The Way of Water,2320250281.00,11720744294.47
3,spider-man: no way home,1921847111.00,9708210681.22
4,The Lion King,1663075401.00,8401025388.15
5,Top Gun: Maverick,1488732821.00,7520333845.28
6,Barbie,1428545028.00,7216295208.94
7,The Super Mario Bros. Movie,1355725263.00,6848446166.04
8,Black Panther,1349926083.00,6819151608.27
9,Star Wars: The Last Jedi,1332698830.00,6732128139.75
10,Jurassic World: Fallen Kingdom,1310466296.00,6619820494.24


### Data limite para as perguntas 5 e 6

Teto = data de lancamento **realizada** mais recente (status `Lançado` e data nao futura).

In [0]:
# primeiro pego a data do lançamento mais recente que realmente aconteceu na base
# considero só status Lançado e data menor ou igual a hoje pra nao pegar filme futuro
# hoje serve só pra filtrar o fim da janela vai ser a maior data encontrada na base

data_max = (filmes
    .filter(
        (F.col("status_filme") == "Lançado")
        & (F.col("data_lancamento") <= F.current_date())
    )
    .agg(F.max("data_lancamento"))
    .collect()[0][0]
)

# se nao tiver nenhum lançamento valido paro aqui pq nao tenho uma data de referencia
if data_max is None:
    raise ValueError("nenhum filme com data de lançamento valida pra definir o periodo")

print("lançamento realizado mais recente:", data_max)

def filmes_lancados_ultimos(anos):
    # uso essa mesma data como teto pras duas perguntas
    # na 5 volto 24 meses e na 6 volto 60 meses
    # pego os lançados depois do inicio da janela e até a data_max incluindo ela
    return filmes.filter(
        (F.col("status_filme") == "Lançado")
        & (F.col("data_lancamento") > F.add_months(F.lit(data_max), -12 * anos))
        & (F.col("data_lancamento") <= F.lit(data_max))
    )

lançamento realizado mais recente: 2026-02-19


### 5. Ator com mais participações nos filmes lançados nos últimos 2 anos

In [0]:
participacoes = (
    filmes_lancados_ultimos(2)
    .join(br_pessoa, "sk_movie_id")
    .join(pessoas, "sk_person_id")
    .filter(F.col("tipo_pessoa") == "Ator")
    .groupBy("nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_participacoes"))
)#
#### aqui conto em quantos filmes cada ator participou dentro dos #####ultimos 2 anos
## o fim desse periodo é o lançamento realizado mais recente da #base
## nao é simplesmente pegar os ultimos 2 anos de hoje

w_ator = Window.orderBy(F.col("qtd_participacoes").desc())

display(
    participacoes
    .withColumn("posicao", F.rank().over(w_ator))
    .filter(F.col("posicao") == 1)
    .select("nome_pessoa", "qtd_participacoes")
    .orderBy("nome_pessoa")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


nome_pessoa,qtd_participacoes
Adlih Torres,12
Andy Dubitsky,12
Cooper Tomlinson,12
Gloria Karel,12
John Simmonds,12
Jonnathon Cripple,12
Sterling L. Pope,12
Tristan Welsh,12


### 6. Produtora com maior lucro nos últimos 5 anos

In [0]:
# aqui somo o lucro dos filmes ligados a cada produtora nos ultimos 5 anos
# se um filme tem duas produtoras, o lucro inteiro entra pras duas
# isso mostra o lucro dos filmes associados, nao quanto cada empresa recebeu de verdade

lucros = (
    filmes_lancados_ultimos(5)
    .join(fato.select("sk_movie_id", "lucro_usd", "lucro_brl"), "sk_movie_id")
    .join(br_empresa, "sk_movie_id")
    .join(empresas, "sk_company_id")
    .groupBy("nome_produtora")
    .agg(
        F.sum("lucro_usd").alias("lucro_total_usd"),
        F.sum("lucro_brl").alias("lucro_total_brl"),
        F.countDistinct("sk_movie_id").alias("qtd_filmes"),
    )
    .filter(F.col("lucro_total_usd").isNotNull())
)
 #a base nao informa a participação de cada produtora
# entao nao vou dividir igualmente e fingir que essa divisão aconteceu # mantenho essa regra explicada e mostro todas as produtoras empatadas em primeiro # filme sem lucro conhecido nao contribui pra soma
w_produtora = Window.orderBy(F.col("lucro_total_usd").desc())

display(
    lucros
    .withColumn("posicao", F.rank().over(w_produtora))
    .filter(F.col("posicao") == 1)
    .drop("posicao")
    .orderBy("nome_produtora")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


nome_produtora,lucro_total_usd,lucro_total_brl,qtd_filmes
Universal Pictures,5772329679.00,29158923373.49,48


In [0]:
# conferencia final: tabelas criadas na gold - CHECK !!!!
display(spark.sql(f"SHOW TABLES IN {gold_schema}"))

database,tableName,isTemporary
gold,bridge_movie_company,false
gold,bridge_movie_genre,false
gold,bridge_movie_person,false
gold,dim_companies,false
gold,dim_genres,false
gold,dim_movies,false
gold,dim_people,false
gold,dim_reviews,false
gold,fact_movies_performance,false
gold,tb_contexto_filmes_ia,false
